# 因子研究

本 Notebook 演示：
1. 运行因子计算流水线
2. 从 `factors.daily_factors` 查询因子数据
3. 因子有效性分析（IC / RankIC）

In [1]:
import os
import sys
sys.path.insert(0, '/app')

import polars as pl
from loguru import logger

print(f'Polars version: {pl.__version__}')

Polars version: 1.39.0


In [2]:
# ── 1. 计算单只股票的因子 ────────────────────────────────────
from app.factors.pipeline import run_factor_pipeline

n = run_factor_pipeline(
    symbols=['000001.SZ'],
    start='2023-01-01',
    end='2024-12-31',
)
print(f'写入因子记录数: {n}')

2026-03-17 12:14:17.042 | INFO     | app.utils.db:make_engine:32 - 数据库引擎已创建: timescaledb:5432/quant_db
2026-03-17 12:14:17.799 | INFO     | app.factors.pipeline:run_factor_pipeline:86 - 000001.SZ ma20: 写入 484 条
2026-03-17 12:14:17.980 | INFO     | app.factors.pipeline:run_factor_pipeline:86 - 000001.SZ ma60: 写入 484 条
2026-03-17 12:14:18.132 | INFO     | app.factors.pipeline:run_factor_pipeline:86 - 000001.SZ rsi14: 写入 484 条
2026-03-17 12:14:18.307 | INFO     | app.factors.pipeline:run_factor_pipeline:86 - 000001.SZ macd: 写入 484 条
2026-03-17 12:14:18.308 | SUCCESS  | app.factors.pipeline:run_factor_pipeline:88 - 因子流水线完成，共写入 1936 条记录


写入因子记录数: 1936


In [3]:
# ── 2. 从 TimescaleDB 查询因子数据 ───────────────────────────
df_factors = pl.read_database_uri(
    query="""
        SELECT time, symbol, factor_name, factor_value
        FROM factors.daily_factors
        WHERE symbol = '000001.SZ'
        ORDER BY time, factor_name
    """,
    uri=os.environ['DATABASE_URL'],
)
df_factors

time,symbol,factor_name,factor_value
"datetime[μs, UTC]",str,str,f64
2023-01-03 00:00:00 UTC,"""000001.SZ""","""ma20""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""ma60""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""macd""",NaN
2023-01-03 00:00:00 UTC,"""000001.SZ""","""rsi14""",NaN
2023-01-04 00:00:00 UTC,"""000001.SZ""","""ma20""",NaN
…,…,…,…
2024-12-30 00:00:00 UTC,"""000001.SZ""","""rsi14""",61.650809
2024-12-31 00:00:00 UTC,"""000001.SZ""","""ma20""",11.6985
2024-12-31 00:00:00 UTC,"""000001.SZ""","""ma60""",11.649167


In [4]:
# ── 3. 转宽格式，方便分析 ────────────────────────────────────
df_wide = df_factors.pivot(
    values='factor_value',
    index=['time', 'symbol'],
    on='factor_name',
).sort('time')
df_wide.tail(10)

time,symbol,ma20,ma60,macd,rsi14
"datetime[μs, UTC]",str,f64,f64,f64,f64
2024-12-18 00:00:00 UTC,"""000001.SZ""",11.511,11.5195,0.004848,53.489145
2024-12-19 00:00:00 UTC,"""000001.SZ""",11.511,11.549167,0.000667,51.256316
2024-12-20 00:00:00 UTC,"""000001.SZ""",11.528,11.577833,-0.000423,52.327858
2024-12-23 00:00:00 UTC,"""000001.SZ""",11.5555,11.606,0.005588,56.135532
2024-12-24 00:00:00 UTC,"""000001.SZ""",11.585,11.630833,0.016999,60.183131
2024-12-25 00:00:00 UTC,"""000001.SZ""",11.6115,11.6545,0.026689,61.92923
2024-12-26 00:00:00 UTC,"""000001.SZ""",11.6375,11.666333,0.027075,59.136419
2024-12-27 00:00:00 UTC,"""000001.SZ""",11.66,11.673167,0.023461,57.734459
2024-12-30 00:00:00 UTC,"""000001.SZ""",11.688,11.668833,0.027048,61.650809


In [10]:
# ── 4. 简单 IC 分析（因子与下期收益的相关性）───────────────────
# 先从 market.daily 取收益率
df_ret = pl.read_database_uri(
    query="""
        SELECT time, symbol, pct_change
        FROM market.daily
        WHERE symbol = '000001.SZ'
        ORDER BY time
    """,
    uri=os.environ['DATABASE_URL'],
)

factor_cols = ['ma20', 'ma60', 'rsi14', 'macd']

df_ic = (
    df_wide
    .join(df_ret, on=['time', 'symbol'])
    # Fix 1: cast Decimal → f64 so pl.corr() works
    .with_columns(pl.col('pct_change').cast(pl.Float64))
    .with_columns(pl.col('pct_change').shift(-1).over('symbol').alias('next_ret'))
    .drop_nulls('next_ret')
    # Fix 2: drop warmup rows where any factor is NaN
    .filter(~pl.any_horizontal(pl.col(c).is_nan() for c in factor_cols if c in df_wide.columns))
)

print(f'有效行数: {len(df_ic)}')
for factor in factor_cols:
    if factor in df_ic.columns:
        ic = df_ic.select(pl.corr(factor, 'next_ret')).item()
        print(f'IC({factor:8s}) = {ic:.4f}')

有效行数: 424
IC(ma20    ) = -0.0972
IC(ma60    ) = -0.0849
IC(rsi14   ) = 0.0239
IC(macd    ) = 0.0020
